# Cadabra2로 검증하는 $n=2$ trace 상쇄

이 노트북은 기존 Python 계산과 분리된 **독립적인 1단계 검산**입니다. 텐서 정준화, 기본 Clifford 곱, 4필드 및 8필드 조합의 여섯 representation moment가 정확히 0이 되는지를 유리수 연산으로 확인합니다.

In [1]:
from cadabra2 import *
print("Cadabra2 kernel is running")

Cadabra2 kernel is running


## 1. 텐서 반대칭 정준화

$A_{mn}=-A_{nm}$이면 $A_{mn}+A_{nm}=0$이어야 합니다.

In [2]:
def assert_zero(ex, label):
    distribute(ex)
    canonicalise(ex)
    collect_terms(ex)
    assert ex == 0, label + " residual: " + str(ex)
    print("PASS:", label)

__cdbkernel__ = create_scope()
{m,n,p,q}::Indices(vector).
A_{m n}::AntiSymmetric.
antisymmetry_check := A_{m n} + A_{n m};
assert_zero(antisymmetry_check, "antisymmetric tensor canonicalisation")

${}A_{m n}+A_{n m}$

PASS: antisymmetric tensor canonicalisation


## 2. 기본 Clifford 곱

Cadabra의 `join_gamma`가 bivector와 vector의 ordered product를 정확히 전개하는지 확인합니다.

In [3]:
\eta{#}::KroneckerDelta.
\Gamma_{#}::GammaMatrix(metric=\eta).
clifford_check := \Gamma^{m n}\Gamma_{p};
join_gamma(clifford_check)
clifford_expected := \Gamma^{m n}_{p} + \Gamma^{m}\eta^{n}_{p} - \Gamma^{n}\eta^{m}_{p};
clifford_residual := @(clifford_check) - @(clifford_expected);
assert_zero(clifford_residual, "Clifford join identity")

${}\Gamma^{m n} \Gamma_{p}$

${}\Gamma^{m n}\,_{p}+\Gamma^{m} \eta^{n}\,_{p}-\,\Gamma^{n} \eta^{m}\,_{p}$

${}0\,$

PASS: Clifford join identity


## 3. 네 필드 조합

여기서 `fD`, `fL`, `fR`, `fLL`, `fRR`, `fLR`은 모든 필드에 공통인 여섯 종류의 배경/연산자 구조입니다. 각 필드에는 **하나의 전체 weight**만 곱합니다. 여섯 구조마다 별도의 임의 weight를 주는 것이 아닙니다.

검사할 조합은 $T+(1/768)B-(1/16)U-(16/3)chi$입니다.

In [4]:
T2   := 256 fD + 16 fL + 16 fR + fLR;
B2   := 65536 fD + 12288 fL + 4096 fR + 1536 fLL + 768 fLR;
U2   := 4096 fD + 512 fL + 256 fR + 32 fLL + 32 fLR;
chi2 := 16 fD + fR;

total4 := @(T2) + 1/768 @(B2) - 1/16 @(U2) - 16/3 @(chi2);
assert_zero(total4, "four-field n=2 moment cancellation")

${}256\,fD+16\,fL+16\,fR+fLR$

${}65536\,fD+12288\,fL+4096\,fR+1536\,fLL+768\,fLR$

${}4096\,fD+512\,fL+256\,fR+32\,fLL+32\,fLR$

${}16\,fD+fR$

${}0\,$

PASS: four-field n=2 moment cancellation


가중치가 우연히 아무 값이나 허용되는 것은 아닙니다. $1/768$을 $1/767$로 바꾸면 잔차가 남아야 합니다.

In [5]:
wrong4 := @(T2) + 1/767 @(B2) - 1/16 @(U2) - 16/3 @(chi2);
distribute(wrong4)
canonicalise(wrong4)
collect_terms(wrong4)
assert wrong4 != 0
wrong4

${}\frac{256}{2301}\,fD+\frac{16}{767}\,fL+\frac{16}{2301}\,fR+\frac{1}{767}\,fLR+\frac{2}{767}\,fLL$

## 4. 기존 여덟 필드 조합

기존 weight `(1, 128, 1/4, 1/4, -12, -12, -1/64, -1/64)`도 같은 방식으로 정확히 검사합니다.

In [6]:
phi2  := fD;
BLL2  := 256 fD + 32 fL + 2 fLL;
BRR2  := 256 fD + 32 fR + 2 fRR;
UL2   := 16 fD + fL;
UR2   := 16 fD + fR;
ULLR2 := 4096 fD + 512 fL + 256 fR + 32 fLL + 32 fLR;
ULRR2 := 4096 fD + 256 fL + 512 fR + 32 fRR + 32 fLR;

total8 := @(T2) + 128 @(phi2) + 1/4 @(BLL2) + 1/4 @(BRR2) - 12 @(UL2) - 12 @(UR2) - 1/64 @(ULLR2) - 1/64 @(ULRR2);
assert_zero(total8, "eight-field n=2 moment cancellation")

${}fD$

${}256\,fD+32\,fL+2\,fLL$

${}256\,fD+32\,fR+2\,fRR$

${}16\,fD+fL$

${}16\,fD+fR$

${}4096\,fD+512\,fL+256\,fR+32\,fLL+32\,fLR$

${}4096\,fD+256\,fL+512\,fR+32\,fRR+32\,fLR$

${}0\,$

PASS: eight-field n=2 moment cancellation


## 현재 검증 범위

이 단계는 핵심 계수 구조와 Cadabra의 텐서/Clifford 처리에 대한 독립 smoke test입니다. 아직 118개 ordered `Box^2` block과 404개 canonical contraction을 Cadabra 안에서 처음부터 재생성한 것은 아닙니다. 다음 단계에서 single-`Box`를 Cadabra로 정의하고 합성·slot 전개·gamma trace까지 독립 구현합니다.